# 🔮 Autoformer con Variables Exógenas — MS Fabric

## Arquitectura corregida

| Variable | Rol | Tratamiento |
|---|---|---|
| `TOTAL` | **Endógena** `y` | Lo que el modelo **predice** |
| `brent_price_usd` | **Exógena futura** | Entra al encoder Y al decoder como covariable |
| `DTWEXBGS` | **Exógena futura** | Entra al encoder Y al decoder como covariable |

### ¿Por qué `futr_exog_list` y no `hist_exog_list`?
- `futr_exog_list`: el modelo recibe los valores de la exógena **durante todo el horizonte** (pasado + futuro). Correcto cuando puedes proyectar o ya conoces los valores futuros de brent y del índice dólar (ej. valores de mercado forward, o se replican los últimos conocidos).
- `hist_exog_list`: el modelo solo ve la exógena en el pasado, no en el horizonte. Útil si no tienes proyecciones futuras.

> Este notebook implementa **ambas opciones** y deja comentada la alternativa para que elijas según tu disponibilidad de datos.

**Fuente:** `INFORME.Ventas_Silver.dataset_autoformer_final_3`  
**Destino:** `INFORME.Ventas_Gold.predictions_autoformer_exog`

## 1. Instalación

In [ ]:
%pip install neuralforecast datasetsforecast pandas==2.0.3 --quiet

## 2. Imports y parámetros

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

from neuralforecast import NeuralForecast
from neuralforecast.models import Autoformer

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# ── Parámetros ────────────────────────────────────────────────────────────────
TARGET_COL   = 'TOTAL'                              # única variable endógena
EXOG_COLS    = ['brent_price_usd', 'DTWEXBGS']     # variables exógenas
GROUP_COL    = 'Station_Out'
DATE_COL     = 'Fecha_venta'

PRED_LEN     = 30    # horizonte de predicción
INPUT_SIZE   = 90    # ventana lookback
MAX_STEPS    = 100
HIDDEN_SIZE  = 64
MIN_ROWS     = INPUT_SIZE + PRED_LEN + 10

TABLA_ORIGEN = 'INFORME.Ventas_Silver.dataset_autoformer_final_3'
TABLA_DESTINO= 'INFORME.Ventas_Gold.predictions_autoformer_exog'

print(f'✅ Config lista')
print(f'   Endógena : {TARGET_COL}')
print(f'   Exógenas : {EXOG_COLS}')
print(f'   Horizonte: {PRED_LEN} días | Lookback: {INPUT_SIZE} días')
print(f'   Dispositivo: {"GPU" if torch.cuda.is_available() else "CPU"}')

## 3. Carga y preprocesamiento

In [ ]:
df_spark = spark.read.table(TABLA_ORIGEN)
df = df_spark.toPandas()

df['Fecha_venta'] = pd.to_datetime(df['Fecha_venta'])
df = df.sort_values(by=['Category', GROUP_COL, DATE_COL])

# Imputar nulos por Station_Out
all_cols = [TARGET_COL] + EXOG_COLS
df[all_cols] = df.groupby(GROUP_COL)[all_cols].ffill().bfill()
df[TARGET_COL] = df[TARGET_COL].astype(float)
df = df.dropna(subset=all_cols)

# Agregar por Station_Out + Fecha (igual que el notebook original)
df = df.groupby([GROUP_COL, DATE_COL]).agg(
    TOTAL=('TOTAL', 'sum'),
    brent_price_usd=('brent_price_usd', 'mean'),
    DTWEXBGS=('DTWEXBGS', 'mean')
).reset_index()

print(f'✅ Datos listos: {len(df):,} filas')
print(f'   Rango: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}')

# Validar grupos con suficientes filas
groups       = df.groupby(GROUP_COL).size().reset_index(name='n_rows')
groups_valid = groups[groups['n_rows'] >= MIN_ROWS]
groups_skip  = groups[groups['n_rows'] <  MIN_ROWS]

print(f'   Grupos entrenables : {len(groups_valid)}')
print(f'   Grupos descartados : {len(groups_skip)}')
if len(groups_skip):
    print(f'   Descartados: {groups_skip[GROUP_COL].tolist()}')

df.head()

## 4. Construcción del DataFrame para `neuralforecast`

`NeuralForecast` requiere estas columnas obligatorias:
- `unique_id` — identificador de serie
- `ds`        — fecha
- `y`         — variable endógena (`TOTAL`)
- + columnas exógenas (`brent_price_usd`, `DTWEXBGS`) con sus nombres originales

In [ ]:
# Filtrar solo grupos válidos
valid_stations = groups_valid[GROUP_COL].tolist()
df_model = df[df[GROUP_COL].isin(valid_stations)].copy()

# Renombrar columnas al formato neuralforecast
df_model['unique_id'] = '_' + df_model[GROUP_COL]   # prefijo igual al notebook original
df_model = df_model.rename(columns={
    DATE_COL   : 'ds',
    TARGET_COL : 'y',
})

# Columnas finales: unique_id, ds, y + exógenas
keep_cols = ['unique_id', 'ds', 'y'] + EXOG_COLS
df_model  = df_model[keep_cols].sort_values(['unique_id', 'ds']).reset_index(drop=True)

print(f'✅ DataFrame para neuralforecast: {df_model.shape}')
print(f'   Series (unique_id): {df_model["unique_id"].nunique()}')
print(f'   Columnas: {list(df_model.columns)}')
df_model.head()

## 5. Proyección de variables exógenas al horizonte futuro

> **Necesario para `futr_exog_list`**: `nf.predict()` requiere valores de las exógenas
> para los `PRED_LEN` días futuros. Aquí se usan tres estrategias, de más simple a más sofisticada:
>
> - **Opción A** *(activa)*: Repetir el último valor conocido (naive forward)
> - **Opción B** *(comentada)*: Promedio móvil de los últimos 30 días
> - **Opción C** *(comentada)*: Ingresar proyecciones reales desde una tabla externa
>
> Si tienes valores de mercado forward del Brent o del índice dólar, usa la Opción C.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Construir df_futr: valores de exógenas en el horizonte futuro
# Formato requerido: unique_id, ds, brent_price_usd, DTWEXBGS
# ─────────────────────────────────────────────────────────────────────────────

futr_rows = []

for uid, grp in df_model.groupby('unique_id'):
    grp = grp.sort_values('ds')
    last_date = grp['ds'].max()

    # Fechas futuras
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=PRED_LEN,
        freq='D'
    )

    # ── OPCIÓN A: Último valor conocido (naive forward) ────────────────────────
    exog_future = {col: grp[col].iloc[-1] for col in EXOG_COLS}

    # ── OPCIÓN B: Promedio móvil 30 días (descomentar para usar) ──────────────
    # exog_future = {col: grp[col].tail(30).mean() for col in EXOG_COLS}

    # ── OPCIÓN C: Proyecciones reales desde tabla externa (descomentar) ────────
    # df_forward = spark.read.table('INFORME.Ventas_Silver.exog_forward_values').toPandas()
    # df_forward['ds'] = pd.to_datetime(df_forward['ds'])
    # exog_future_df = df_forward[df_forward['ds'].isin(future_dates)].set_index('ds')

    for fd in future_dates:
        row = {'unique_id': uid, 'ds': fd}
        row.update({col: exog_future[col] for col in EXOG_COLS})
        futr_rows.append(row)

df_futr = pd.DataFrame(futr_rows)
df_futr['ds'] = pd.to_datetime(df_futr['ds'])

print(f'✅ DataFrame de exógenas futuras: {df_futr.shape}')
print(f'   Fechas futuras por serie: {PRED_LEN}')
print(f'   Estrategia: Último valor conocido (Opción A)')
df_futr.head()

## 6. Definición del modelo Autoformer con exógenas

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  futr_exog_list  →  el modelo recibe brent y DTWEXBGS en todo el horizonte
#                     (pasado durante entrenamiento + futuro durante predicción)
#
#  Alternativa con hist_exog_list (solo pasado, sin proyecciones futuras):
#
#  Autoformer(
#      h=PRED_LEN,
#      input_size=INPUT_SIZE,
#      hist_exog_list=EXOG_COLS,   # ← solo ve exógenas en ventana histórica
#      max_steps=MAX_STEPS,
#      hidden_size=HIDDEN_SIZE,
#  )
#  Con esta opción NO necesitas df_futr en nf.predict()
# ─────────────────────────────────────────────────────────────────────────────

models = [
    Autoformer(
        h            = PRED_LEN,
        input_size   = INPUT_SIZE,
        futr_exog_list = EXOG_COLS,   # ← brent y DTWEXBGS como exógenas futuras
        max_steps    = MAX_STEPS,
        hidden_size  = HIDDEN_SIZE,
    )
]

nf = NeuralForecast(models=models, freq='D')

print('✅ Modelo definido')
print(f'   Arquitectura     : Autoformer')
print(f'   Endógena (y)     : {TARGET_COL}')
print(f'   Exógenas futuras : {EXOG_COLS}')
print(f'   Horizonte        : {PRED_LEN} días')
print(f'   Lookback         : {INPUT_SIZE} días')
print(f'   max_steps        : {MAX_STEPS}')

## 7. Entrenamiento

In [ ]:
print('🚀 Entrenando Autoformer...')
print(f'   Series: {df_model["unique_id"].nunique()}')
print(f'   Filas totales: {len(df_model):,}')

# df_model contiene: unique_id, ds, y (TOTAL), brent_price_usd, DTWEXBGS
# neuralforecast detecta automáticamente las columnas exógenas declaradas en futr_exog_list
nf.fit(df=df_model)

print('✅ Entrenamiento completado')

## 8. Predicción

In [ ]:
print('🔮 Generando predicciones...')

# futr_df: valores de exógenas en el horizonte futuro (obligatorio con futr_exog_list)
forecast = nf.predict(futr_df=df_futr)

# Renombrar columna de salida
forecast = forecast.rename(columns={'Autoformer': 'pred_TOTAL'})

# Recuperar Station_Out desde unique_id
forecast['Station_Out'] = forecast['unique_id'].str.lstrip('_')
forecast = forecast.drop(columns=['unique_id'])
forecast = forecast.rename(columns={'ds': 'Fecha_venta'})

# Ordenar columnas
forecast = forecast[['Station_Out', 'Fecha_venta', 'pred_TOTAL']]

print(f'✅ Predicciones generadas: {len(forecast):,} filas')
print(f'   Estaciones: {forecast["Station_Out"].nunique()}')
print(f'   Rango predicho: {forecast["Fecha_venta"].min().date()} → {forecast["Fecha_venta"].max().date()}')
forecast.head(10)

## 9. Cross-validation con exógenas

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# La CV con futr_exog_list requiere que se proporcionen las exógenas
# para cada ventana de validación. neuralforecast lo gestiona internamente
# tomando los valores reales del df (son conocidos en el pasado).
# ─────────────────────────────────────────────────────────────────────────────

print('🔄 Ejecutando cross-validation...')

try:
    df_cv = nf.cross_validation(
        df       = df_model,
        n_windows= 3,
        step_size= PRED_LEN,
    )
    df_cv = df_cv.rename(columns={
        'Autoformer': 'pred_TOTAL',
        'y'         : 'TOTAL'
    })
    df_cv['Station_Out'] = df_cv['unique_id'].str.lstrip('_')
    df_cv = df_cv.drop(columns=['unique_id'])
    df_cv = df_cv.rename(columns={'ds': 'Fecha_venta'})

    # Métricas por estación
    def mape(yt, yp):
        mask = np.abs(yt) > 1e-6
        return np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.sum() else np.nan

    print('\n📊 Métricas de Cross-Validation (TOTAL):')
    print(f'   {"Station_Out":<20} {"MAE":>10} {"RMSE":>10} {"MAPE":>10}')
    print('   ' + '-' * 52)

    for sta, grp in df_cv.groupby('Station_Out'):
        yt = grp['TOTAL'].values
        yp = grp['pred_TOTAL'].values
        mae  = np.mean(np.abs(yt - yp))
        rmse = np.sqrt(np.mean((yt - yp)**2))
        mp   = mape(yt, yp)
        print(f'   {sta:<20} {mae:>10.2f} {rmse:>10.2f} {mp:>9.2f}%')

except Exception as e:
    print(f'⚠️  Error en CV: {e}')
    df_cv = pd.DataFrame()

## 10. Guardado en Lakehouse

In [ ]:
# ── Predicciones futuras ───────────────────────────────────────────────────────
spark_forecast = spark.createDataFrame(forecast)
(
    spark_forecast
    .write.format('delta')
    .mode('overwrite')
    .partitionBy('Station_Out')
    .saveAsTable(TABLA_DESTINO)
)
print(f'✅ Predicciones guardadas en: {TABLA_DESTINO}')

# ── Resultados de CV ──────────────────────────────────────────────────────────
if not df_cv.empty:
    cv_table = TABLA_DESTINO.replace('predictions', 'cv_results')
    spark.createDataFrame(df_cv).write.format('delta').mode('overwrite').saveAsTable(cv_table)
    print(f'✅ CV guardado en: {cv_table}')

# Vista final
spark.sql(f'SELECT * FROM {TABLA_DESTINO} ORDER BY Station_Out, Fecha_venta LIMIT 15').show(truncate=False)

## Resumen

In [ ]:
print('=' * 60)
print('  AUTOFORMER — RESUMEN DE EJECUCIÓN')
print('=' * 60)
print(f'  Variable endógena  : {TARGET_COL}')
print(f'  Variables exógenas : {EXOG_COLS}')
print(f'  Tipo de exógenas   : futr_exog_list (futuras conocidas)')
print(f'  Proyección exógenas: Último valor conocido (Opción A)')
print()
print(f'  Estaciones entrenadas : {forecast["Station_Out"].nunique()}')
print(f'  Horizonte predicho    : {PRED_LEN} días')
print(f'  Lookback              : {INPUT_SIZE} días')
print()
print(f'  Predicciones → {TABLA_DESTINO}')
print('=' * 60)